<a href="https://colab.research.google.com/github/mukulhayaran/pyTorch-Learning/blob/main/pyTorch_dataloaders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
 import torch
 import torch.nn
 import torchvision
 from torchvision import transforms
 from torch.utils.data import Dataset,DataLoader
 from torchvision.datasets import ImageFolder #stream data from images stored in folders

 import os #allows access to files
 import numpy as np
 from PIL import Image # helps load images
 from collections import Counter #gives count of unique items in an iterable

In [3]:
from google.colab import files

uploaded = files.upload()

import zipfile
import os

zip_file = list(uploaded.keys())[0]  # get uploaded file name

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("Extracted to /content")

IndexError: list index out of range

In [4]:
# Download the dataset directly
!wget -O cats_vs_dogs.zip "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

# Unzip it silently into a folder named 'dataset_folder'
!unzip -q cats_vs_dogs.zip -d dataset_folder

--2026-03-17 08:31:04--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.218.186.60, 2600:1409:3c00:c80::317f, 2600:1409:3c00:c8c::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.218.186.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘cats_vs_dogs.zip’

cats_vs_dogs.zip    100%[===================>] 786.67M   111MB/s    in 7.6s    

2026-03-17 08:31:11 (104 MB/s) - ‘cats_vs_dogs.zip’ saved [824887076/824887076]



In [5]:
class DogsVsCat(Dataset):
  def __init__(self,path_to_folder):
    path_to_cats=os.path.join(path_to_folder,"Cat")
    path_to_dogs=os.path.join(path_to_folder,"Dog")

    cat_files=os.listdir(path_to_cats)
    dog_files=os.listdir(path_to_dogs)

    path_to_cat_files=[os.path.join(path_to_cats,file) for file in cat_files]
    path_to_dog_files=[os.path.join(path_to_dogs,file) for file in dog_files]

    self.training_files=path_to_dog_files+path_to_cat_files
    self.dog_label=0
    self.cat_label=1

    self.transform=transforms.ToTensor()

  def __len__(self):
    return len(self.training_files)

  def __getitem__(self,idx):
    path_to_image = self.training_files[idx]
    if "Dog" in path_to_image:
      label=self.dog_label
    else:
      label=self.cat_label

    image=Image.open(path_to_image).convert("RGB")
    image=self.transform(image)
    return image


path_to_folder="/content/dataset_folder/PetImages"
dataset=DogsVsCat(path_to_folder)


Loading Dataset class into DataLoader

In [7]:
dogsvscatloader=DataLoader(dataset,batch_size=16, shuffle=False)
for images,labels in dogsvscatloader:
  print(images.shape)
  print(labels.shape)

# this will give error because the tensors are not of equal size

RuntimeError: stack expects each tensor to be equal size, but got [3, 375, 500] at entry 0 and [3, 235, 240] at entry 1

Torchvision Transforms:

In [8]:
img_transforms=transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

    ]
)
#mean and std's of r,g ,b

class DogsVsCat(Dataset):
  def __init__(self,path_to_folder):
    path_to_cats=os.path.join(path_to_folder,"Cat")
    path_to_dogs=os.path.join(path_to_folder,"Dog")

    cat_files=os.listdir(path_to_cats)
    dog_files=os.listdir(path_to_dogs)

    path_to_cat_files=[os.path.join(path_to_cats,file) for file in cat_files]
    path_to_dog_files=[os.path.join(path_to_dogs,file) for file in dog_files]

    self.training_files=path_to_dog_files+path_to_cat_files
    self.dog_label=0
    self.cat_label=1

    self.transform=transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

    ])

  def __len__(self):
    return len(self.training_files)

  def __getitem__(self,idx):
    path_to_image = self.training_files[idx]
    if "Dog" in path_to_image:
      label=self.dog_label
    else:
      label=self.cat_label

    image=Image.open(path_to_image).convert("RGB")
    image=self.transform(image)
    return image,label


path_to_folder="/content/dataset_folder/PetImages"
dataset=DogsVsCat(path_to_folder)

dataset=DogsVsCat(path_to_folder)


dogsvscatloader=DataLoader(dataset,batch_size=16, shuffle=True)
for images,labels in dogsvscatloader:
  print(images.shape)
  print(labels)
  break

  # We use shuffle = True, to get a mix of dog and cat images in the batch


torch.Size([16, 3, 224, 224])
tensor([1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1])


Test/Train split:


In [10]:
num_train_samples=int(0.9*len(dataset))
num_test_samples=len(dataset)-num_train_samples

num_train_samples,num_test_samples

train_dataset,test_dataset=torch.utils.data.random_split(dataset,[num_train_samples,num_test_samples])

dogsvscatloader_train=DataLoader(train_dataset,batch_size=16, shuffle=True)
dogsvscatloader_test=DataLoader(test_dataset,batch_size=16, shuffle=True)

(22501, 2501)

In [1]:
#we can use ImageFolder of Datasets module in Torchvision.
#it does the same thing as the above code but with very few lines of code.

dataset=ImageFolder(path_to_folder)
dataset

NameError: name 'ImageFolder' is not defined